## Función de recomendación
Función que recomienda las 10 mejores rutas según el nivel del ciclista: Beginner, Intermediate o Advanced.

In [10]:
def recommend_routes(level, df):
    # Filtrar rutas según el nivel ingresado
    if level == "Beginner":
        routes = df[df["difficulty_score"] <= 0.33]
    elif level == "Intermediate":
        routes = df[(df["difficulty_score"] > 0.33) & (df["difficulty_score"] <= 0.66)]
    elif level == "Advanced":
        routes = df[df["difficulty_score"] > 0.66]
    else:
        print("Nivel no válido. Usa: Beginner, Intermediate o Advanced")
        return None

    # Retornar las 10 mejores rutas ordenadas por dificultad
    return routes.sort_values("difficulty_score")[
        ["name", "distance_km", "average_speed_kmh", "elevation_m", "speed_range", "difficulty_score"]
    ].head(10)

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv('dataset/strava_data.csv')
df = df[df['sport_type'] == 'Ride']
df = df[["name", "distance", "moving_time", "total_elevation_gain", "average_speed", "sport_type", "date"]]
df = df.drop_duplicates()
df = df[df["average_speed"] > 5]
df = df[df["average_speed"] < 60]

speed_conditions = [
    df["average_speed"] < 20,
    (df["average_speed"] >= 20) & (df["average_speed"] < 30),
    df["average_speed"] >= 30
]
df = df.assign(
    speed_range=np.select(speed_conditions, ["< 20 km/h", "20–30 km/h", "> 30 km/h"], default="unknown"),
    level=np.select(speed_conditions, ["Beginner", "Intermediate", "Advanced"], default="unknown")
)
df["distance"] = df["distance"] * 1.60934
df["average_speed"] = df["average_speed"] * 1.60934
df["total_elevation_gain"] = df["total_elevation_gain"] * 0.3048
df["date"] = pd.to_datetime(df["date"])
df = df.rename(columns={
    "distance": "distance_km",
    "average_speed": "average_speed_kmh",
    "total_elevation_gain": "elevation_m"
})
df = df.assign(
    elev_norm=(df["elevation_m"] - df["elevation_m"].min()) / (df["elevation_m"].max() - df["elevation_m"].min()),
    dist_norm=(df["distance_km"] - df["distance_km"].min()) / (df["distance_km"].max() - df["distance_km"].min()),
    speed_norm=(df["average_speed_kmh"] - df["average_speed_kmh"].min()) / (df["average_speed_kmh"].max() - df["average_speed_kmh"].min())
)
df = df.assign(
    difficulty_score=(
        df["elev_norm"] * 0.5 +
        df["dist_norm"] * 0.3 +
        df["speed_norm"] * 0.2
    )
)
print("Dataset listo:", df.shape)

Dataset listo: (495, 13)


## Resultados por nivel
Se obtienen resultados de acuerdo al nivel.

In [12]:
# Pruebas
print("\n=== Rutas para Beginner ===")
print(recommend_routes("Beginner", df))
print("\n=== Rutas para Intermediate ===")
print(recommend_routes("Intermediate", df))
print("\n=== Rutas para Advanced ===")
print(recommend_routes("Advanced", df))


=== Rutas para Beginner ===
                name  distance_km  average_speed_kmh  elevation_m speed_range  \
358   Afternoon Ride     0.160934           9.656040     0.000000   < 20 km/h   
557       Night Ride     1.609340           9.656040     0.000000   < 20 km/h   
1967    Evening Ride     1.657620          10.412430     1.999488   < 20 km/h   
944     Morning Ride     3.234773           8.529502    30.001464   < 20 km/h   
130     Morning Ride     0.128747          11.796462     4.998720   < 20 km/h   
382     Evening Ride     3.540548          10.653831    14.999208   < 20 km/h   
1065           jTRee     5.165981           9.929628    24.999696   < 20 km/h   
657   Afternoon Ride     4.602712          10.782578    13.999464   < 20 km/h   
861    Wheelies baby     5.632690          11.265380     0.000000   < 20 km/h   
605   Afternoon Ride     1.721994          12.874720     0.000000   < 20 km/h   

      difficulty_score  
358           0.006786  
557           0.009480  
1967